In [1]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer 
import matplotlib.pyplot as plt
import numpy as np

/Users/Hetansh/Github/research_project_1/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
model= GPT2LMHeadModel.from_pretrained('gpt2')
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(device)

cpu


In [4]:
#Understanding tokenization
test_prompt = "The Eiffel Tower is located in"
tokens= tokenizer.tokenize(test_prompt)
token_ids= tokenizer.encode(test_prompt)
for i, (token,id) in enumerate(zip(tokens,token_ids)):
    print(f"Position {i}: '{token}' -> ID {id} ")

Position 0: 'The' -> ID 464 
Position 1: 'ĠE' -> ID 412 
Position 2: 'iff' -> ID 733 
Position 3: 'el' -> ID 417 
Position 4: 'ĠTower' -> ID 8765 
Position 5: 'Ġis' -> ID 318 
Position 6: 'Ġlocated' -> ID 5140 
Position 7: 'Ġin' -> ID 287 


### Running forward pass

In [5]:
inputs= tokenizer(test_prompt, return_tensors='pt').to(device)


with torch.no_grad():
    outputs= model(
        **inputs,
        output_hidden_states= True,
        output_attentions=True
    )
    

In [6]:
print("Output structure:")
print(f"  - logits shape: {outputs.logits.shape}")
print(f"    (batch_size, sequence_length, vocab_size)")
print(f"  - hidden_states: tuple of {len(outputs.hidden_states)} tensors")
print(f"    (one for embeddings + one per layer)")
print(f"  - Each hidden state shape: {outputs.hidden_states[0].shape}")
print(f"    (batch_size, sequence_length, hidden_size=768)")


Output structure:
  - logits shape: torch.Size([1, 8, 50257])
    (batch_size, sequence_length, vocab_size)
  - hidden_states: tuple of 13 tensors
    (one for embeddings + one per layer)
  - Each hidden state shape: torch.Size([1, 8, 768])
    (batch_size, sequence_length, hidden_size=768)


In [7]:
layer_idx=6
last_token_idx=-1

In [8]:
hidden_state= outputs.hidden_states[layer_idx + 1][0, last_token_idx,:]
print(f"\nHidden state at layer {layer_idx} for last token:")
print(f"Shape: {hidden_state.shape}")
print(f"Mean: {hidden_state.mean():.2f}")
print(f"Std: {hidden_state.std():.2f}")


#top-k predictions

logits= outputs.logits[0,-1,:]
probs= torch.softmax(logits,dim=-1)




Hidden state at layer 6 for last token:
Shape: torch.Size([768])
Mean: 0.06
Std: 2.89


In [9]:
top_k=10

top_probs, top_indices = torch.topk(probs, top_k)

print(f"\n Top {top_k} predictions:")
for i,(prob,idx) in enumerate(zip(top_probs,top_indices)):
    token=tokenizer.decode([idx.item()])
    print(f" {i+1}.'{token}'- {prob.item()*100:.2f}%")


#helper fn

def get_model_prediction(model,tokenizer,prompt,device):
    """
    Get the model's top prediction and probability for a given prompt.
    Returns:
    dict with keys: 'top_token','top_prob','target_prob'(if target provided)
    """

    inputs= tokenizer(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        outputs=model(**inputs)

    logits= outputs.logits[0,-1,:]
    probs= torch.softmax(logits,dim=-1)

    top_prob, top_idx = torch.max(probs, dim=-1)

    top_token= tokenizer.decode([top_idx.item()])
    return{
        'top_token':top_token,
        'top_prob':top_prob.item(),
        'all_probs':probs.cpu().numpy()
    }


result = get_model_prediction(model,tokenizer, "The capital of France is", device)

print(f"\n Testing helper function:")
print(f" Prompt: 'The capital of France is'")
print(f"Top prediction: '{result['top_token']}' with probability {result['top_prob']*100:.2f}%")




 Top 10 predictions:
 1.' the'- 39.74%
 2.' a'- 7.54%
 3.' central'- 2.64%
 4.' downtown'- 2.19%
 5.' an'- 2.18%
 6.' London'- 1.72%
 7.' front'- 1.64%
 8.' New'- 1.17%
 9.' Manhattan'- 1.15%
 10.' one'- 0.95%

 Testing helper function:
 Prompt: 'The capital of France is'
Top prediction: ' the' with probability 8.46%
